# Liver dataset selection — audit (`_repaired`)

Analogicznie do `kidney_dataset_repaired.ipynb`: audytuję już pobrany korpus liver (`data/liver_workspace/configs/datasets/liver/`, 18 datasetów) tą samą metodyką co `brain_dataset.ipynb` — obiektywna detekcja duplikatów technicznych, warianty kalibracyjne, fragmenty mikroanatomiczne, ekstremalnie małe akwizycje.

**Ważna różnica względem kidney: liver nigdy nie przeszedł ręcznego przeglądu pod kątem pseudoreplikacji** — `filter.json` ma `exclude_dataset_ids: []`. Obecne 18 datasetów zostało zaakceptowanych wprost z zapytania, bez etapu ręcznego wykluczania duplikatów (w przeciwieństwie do kidney, gdzie taki przegląd dał 30 wykluczeń). To robi ten audyt bardziej istotnym dla liver niż dla kidney.

**Świadomie POMINIĘTE (na wyraźną prośbę):** analiza wspólnego zakresu m/z między narządami — zostawiam bez zmian dotychczasowy zakres liver (`mz_min=200, mz_max=1400`).

**Ograniczenia jak w pozostałych dwóch notebookach:** wyłącznie `DatasetExplorer`, zero zmian w bibliotece, zero nowych pobrań surowych danych.

In [1]:
import os
from pathlib import Path

current_path = Path.cwd().resolve()
repository_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "pyproject.toml").is_file()
)
os.chdir(repository_root)

repository_root

PosixPath('/home/max/repositories/MSIAutoEncoderWrapper')

In [2]:
import json
import re

import pandas as pd
from IPython.display import display

from msi_dataset_manager.exploration import DatasetExplorer

# REMARK: date i download DB is 12.08.2026 (DD, MM, YYYY) -- same cache as the other two notebooks.
explorer = DatasetExplorer(
    source="metaspace",
    cache_dir="assets/local/datasets/metaspace",
    refresh_cache=False,
)

## 1. Szeroka pula kandydatów (ten sam filtr biologiczny co dotychczasowy `liver_dataset.1.ipynb`)

`condition="Wildtype"`, bez `mz_min`/`mz_max` — tak samo jak przy kidney, żeby audyt objął też rekordy odrzucane dziś przez filtr `200–1400`.

In [3]:
broad_filters = {
    "organism": "Mouse",
    "organism_part": "Liver",
    "condition": "Wildtype",
    "polarity": "Negative",
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
}
results = explorer.filter(broad_filters)
print(f"Found {len(results)} datasets")
display(results[["dataset_id", "name", "analyzer_type", "ionisation_source", "mz_min", "mz_max", "pixel_count"]])

METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

Found 82 datasets


,dataset_id,name,analyzer_type,ionisation_source,mz_min,mz_max,pixel_count
0,2026-06-22_15h21m26s,WT_D12_3-neg,TOF,DESI,7.500978e+01,1199.833220,13964
1,2026-06-22_15h19m59s,WT_D12_2-neg,TOF,DESI,7.500978e+01,1199.833220,5142
2,2026-06-22_15h19m49s,WT_D12_1-neg,TOF,DESI,7.500978e+01,1199.833220,9002
3,2026-06-22_15h19m03s,WT_D5_3-neg,TOF,DESI,7.500978e+01,1199.833220,9488
4,2026-06-22_15h17m46s,WT_D5_2-neg,TOF,DESI,7.500978e+01,1199.833220,7172
...,...,...,...,...,...,...,...
77,2022-04-14_15h07m57s,2022-04-14_ME_DKFZACLY_S1_W4_DANneg_s10a33_100...,Orbitrap,AP-SMALDI5,9.900769e+01,404.029861,10000
78,2022-04-14_14h45m46s,2022-04-13_ME_DKFZACLY_S1_W8_DANneg_s10a33_100...,Orbitrap,AP-SMALDI5,9.900768e+01,404.029265,10000
79,2017-02-23_09h51m18s,Mouse liver_DMAN_200x200_25um_rcal,Orbitrap,MALDI,2.499993e+02,999.884929,40000
80,2017-02-22_15h01m27s,210217_mouseliver_DMAN_negative_200x200_25um-2...,Orbitrap,MALDI,2.500000e+02,999.881226,40000


## 2. Obiektywna detekcja duplikatów technicznych

Ta sama reguła co w brain/kidney: identyczny `pixel_count` **oraz** identyczne `mz_min`/`mz_max` (zaokrąglone do 3 miejsc). Dodatkowo, tak jak w kidney, każdy klaster dzielę na `high_confidence_duplicate` (nazwy zbiegają się po usunięciu tokenów technicznych) vs `ambiguous_shared_template` (nie zbiegają się — możliwy współdzielony protokół akwizycji, a nie duplikat).

**To właśnie tutaj po raz pierwszy natrafiłem na `ambiguous_shared_template`:** pięć datasetów `2022-04-1*_ME_DKFZACLY_S{1,2,3}_W{4,8}_DANneg_..._100x100_100-400_NCE25` ma identyczny `pixel_count=10000` i (po zaokrągleniu) identyczny `mz_min`/`mz_max` — ale różnią się identyfikatorem slajdu/studzienki (`S1`/`S2`/`S3` × `W4`/`W8`), czyli to prawdopodobnie **pięć różnych fizycznych próbek zmierzonych wg tego samego, stałego protokołu** (ta sama siatka 100×100, to samo nominalne okno masy 100–400 Da), a nie ten sam skan zgłoszony wielokrotnie. Gdybym polegał wyłącznie na `(pixel_count, mz_min, mz_max)`, wykluczyłbym błędnie 4 z 5 niezależnych próbek. Sprawdzenie zbieżności nazw po usunięciu tokenów technicznych temu zapobiega.

(Ciekawostka przy okazji: szósty dataset z tej samej serii, `..._S1_W4_...`, ma `mz_max` różniący się od pozostałych o jedną tysięczną — po zaokrągleniu do 3 miejsc wypada **poza** klastrem. To pokazuje, że sama metoda zaokrąglania może też dawać fałszywe negatywy na granicy progu, nie tylko fałszywe pozytywy.)

In [4]:
results["mz_min_r"] = results["mz_min"].round(3)
results["mz_max_r"] = results["mz_max"].round(3)
results["cluster_id"] = results.groupby(["pixel_count", "mz_min_r", "mz_max_r"]).ngroup()
cluster_size = results.groupby("cluster_id")["dataset_id"].transform("count")
results["is_duplicate_cluster"] = cluster_size > 1


def strip_technical_tokens(name: str) -> str:
    s = str(name).lower()
    s = re.sub(r"^\d{4}-\d{2}-\d{2}[_ ]", "", s)
    s = re.sub(r"^\d{8}_+", "", s)
    s = re.sub(r"[-_]?\d+ ?ppm\b", "", s)
    s = re.sub(r"\btic\b", "", s)
    s = re.sub(r"_(?:aq_ml|aq|ml)$", "", s)
    s = re.sub(r"-total ion count$", "", s)
    s = re.sub(r" - root mean square$", "", s)
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    return s


results["name_residual"] = results["name"].apply(strip_technical_tokens)

cluster_confidence = {}
for cluster_id, group in results[results["is_duplicate_cluster"]].groupby("cluster_id"):
    residuals = set(group["name_residual"])
    cluster_confidence[cluster_id] = (
        "high_confidence_duplicate" if len(residuals) == 1 else "ambiguous_shared_template"
    )
results["cluster_confidence"] = results["cluster_id"].map(cluster_confidence)

TECH_SUFFIX_PENALTY = re.compile(r"(?:_ml$|_v$|ppm$|-total ion count$)", re.IGNORECASE)


def pick_keeper(group: "pd.DataFrame") -> str:
    scored = group.assign(
        penalty=group["name"].str.lower().str.contains(TECH_SUFFIX_PENALTY, regex=True).astype(int)
    )
    return scored.sort_values(["penalty", "dataset_id"]).iloc[0]["dataset_id"]


keepers = {
    cluster_id: pick_keeper(group)
    for cluster_id, group in results[results["cluster_confidence"] == "high_confidence_duplicate"].groupby("cluster_id")
}
results["duplicate_excluded"] = results.apply(
    lambda row: row.get("cluster_confidence") == "high_confidence_duplicate"
    and row["dataset_id"] != keepers[row["cluster_id"]],
    axis=1,
)

n_clusters = results.loc[results["is_duplicate_cluster"], "cluster_id"].nunique() if results["is_duplicate_cluster"].any() else 0
print("duplicate clusters found (any confidence):", n_clusters)
print("confirmed (high-confidence) duplicate exclusions:", int(results["duplicate_excluded"].sum()))
display(
    results.loc[
        results["is_duplicate_cluster"],
        ["cluster_id", "dataset_id", "name", "pixel_count", "mz_min_r", "mz_max_r", "cluster_confidence", "duplicate_excluded"],
    ].sort_values(["cluster_confidence", "cluster_id"])
)

duplicate clusters found (any confidence): 6
confirmed (high-confidence) duplicate exclusions: 5


,cluster_id,dataset_id,name,pixel_count,mz_min_r,mz_max_r,cluster_confidence,duplicate_excluded
56,28,2022-04-16_08h23m55s,2022-04-14_ME_DKFZACLY_S3_W8_DANneg_s10a33_100...,10000,99.008,404.029,ambiguous_shared_template,False
57,28,2022-04-16_08h22m38s,2022-04-14_ME_DKFZACLY_S3_W4_DANneg_s10a33_100...,10000,99.008,404.029,ambiguous_shared_template,False
75,28,2022-04-16_08h17m46s,2022-04-14_ME_DKFZACLY_S2_W8_DANneg_s10a33_100...,10000,99.008,404.029,ambiguous_shared_template,False
76,28,2022-04-16_08h18m19s,2022-04-14_ME_DKFZACLY_S2_W4_DANneg_s10a33_100...,10000,99.008,404.029,ambiguous_shared_template,False
78,28,2022-04-14_14h45m46s,2022-04-13_ME_DKFZACLY_S1_W8_DANneg_s10a33_100...,10000,99.008,404.029,ambiguous_shared_template,False
22,47,2026-02-24_17h39m55s,2026_02_24_Rep-01,28745,50.000,1000.000,ambiguous_shared_template,False
23,47,2026-02-23_17h18m48s,2026_02_19_Rep-01,28745,50.000,1000.000,ambiguous_shared_template,False
10,55,2026-03-31_13h24m27s,id2_calnaf_quadraticen,34049,50.000,1000.000,ambiguous_shared_template,False
12,55,2026-03-30_16h51m11s,2026_03_25_calnaf_quadraticen,34049,50.000,1000.000,ambiguous_shared_template,False
44,0,2024-06-28_07h10m21s,liver rn lip tic-50ppm,110,399.603,999.609,high_confidence_duplicate,True


### Wariant kalibracyjny `null_mz_shift`

Sprawdzone dla kompletności (jak w brain i kidney) — w puli liver **nie występuje** żaden dataset nazwany w ten sposób.

In [5]:
results["mz_shift_qc_variant"] = results["name"].str.contains("null_mz_shift", case=False, na=False)
print("mz-shift QC variants:", int(results["mz_shift_qc_variant"].sum()))

mz-shift QC variants: 0


## 3. Morfologia i jakość

**Morfologia (Poziom 4):** dla liver istotne byłyby fragmenty typu płat (`lobe`), strefowanie okołowrotne/okołożylne (`periportal`/`pericentral`), torebka (`capsule`), wnęka wątroby (`portal`). W tej puli **0** trafień — heurystyka nie znajduje żadnego jawnie nazwanego fragmentu wątroby (w przeciwieństwie do kidney, gdzie `glomeruli` dało 2 trafienia). To zgodne z tym, że nazwy datasetów liver (`Lipids*`, `Tissue*`, `NEDC_imaging_liver`, `liver storage day N`, serie `Norton`/`DKFZACLY`) opisują całe przekroje, nie regiony — ale heurystyka nie dowodzi tego w 100%, tylko brak sygnału przeciwnego w nazwie.

**Jakość / liczba pikseli (Poziom 2):** liver ma systemowo dużo mniejsze akwizycje niż kidney/brain — mediana ok. 12 000 pikseli, ale 5. percentyl to tylko ok. 465, a minimum w puli to 110 pikseli. Sztywny próg `500` z brain odciąłby tu część **już zaakceptowanych** przez Ciebie datasetów (np. serie `Lipids*`/`Tissue*` IR-MALDESI mają regularnie 500–1000 pikseli — to ich normalna skala, nie wada). Kalibruję więc próg osobno dla liver na `200` — łapie tylko skrajne przypadki (które i tak już są częścią klastra duplikatów z sekcji 2: `liver rn lip tic*` = 110 px, `liver storage day 5 tic` = 192 px), a nie normalną, mniejszą skalę tego narządu.

In [6]:
REGIONAL_TOKENS = re.compile(r"(?:\blobe\b|periportal|pericentral|zonation|capsule|\bportal\b)", re.IGNORECASE)
results["morphology_hint"] = results["name"].apply(
    lambda n: "regional_or_microregion" if REGIONAL_TOKENS.search(str(n)) else "whole_section_likely"
)
print("regional/microregion hint count (heuristic, advisory):",
      int((results["morphology_hint"] == "regional_or_microregion").sum()))

LOW_PIXEL_THRESHOLD = 200  # calibrated to liver's own scale, see markdown above -- NOT the brain/kidney value of 500
results["low_pixel_flag"] = results["pixel_count"] < LOW_PIXEL_THRESHOLD
print("low pixel_count (<200) flagged:", int(results["low_pixel_flag"].sum()))
display(results.loc[results["low_pixel_flag"], ["dataset_id", "name", "pixel_count", "duplicate_excluded"]])

regional/microregion hint count (heuristic, advisory): 0
low pixel_count (<200) flagged: 3


,dataset_id,name,pixel_count,duplicate_excluded
44,2024-06-28_07h10m21s,liver rn lip tic-50ppm,110,True
46,2024-06-26_03h54m35s,liver rn lip tic,110,False
48,2024-06-25_11h23m24s,liver storage day 5 tic,192,False


## 4. Zestawienie z istniejącą selekcją

`data/liver_workspace/configs/datasets/liver/filter.json` ma `exclude_dataset_ids: []` — liver nigdy nie przeszedł ręcznego przeglądu pod kątem duplikatów. Sprawdzam, czy dzisiejszy korpus 18 datasetów mimo to trafił przypadkiem na któryś z obiektywnie znalezionych problemów.

In [7]:
existing_filter = json.load(open("data/liver_workspace/configs/datasets/liver/filter.json"))
existing_selection = json.load(open("data/liver_workspace/configs/datasets/liver/selection.json"))
existing_excluded_ids = set(existing_filter.get("exclude_dataset_ids", []))
existing_selected_ids = set(existing_selection["dataset_ids"])
print(f"existing selection: {len(existing_selected_ids)} selected, {len(existing_excluded_ids)} manually excluded")

new_findings = set(
    results.loc[results["duplicate_excluded"] | results["mz_shift_qc_variant"], "dataset_id"]
)
print("objective findings currently in the downloaded 18:", sorted(new_findings & existing_selected_ids))
print("objective findings not in the downloaded 18 (only relevant if the corpus is widened later):",
      sorted(new_findings - existing_selected_ids))

existing selection: 18 selected, 0 manually excluded
objective findings currently in the downloaded 18: []
objective findings not in the downloaded 18 (only relevant if the corpus is widened later): ['2023-06-02_11h54m37s', '2024-06-25_11h38m02s', '2024-06-26_03h47m05s', '2024-06-28_07h10m21s', '2024-07-01_09h59m02s']


## 5. Poprawiona ("repaired") konfiguracja

Piszę do równoległego katalogu `liver_repaired/`, nie nadpisuję `liver/`. Zakres m/z zostaje niezmieniony (`200–1400`). Ponieważ (patrz sekcja 4) obecne 18 datasetów **nie zawiera** żadnego z obiektywnie znalezionych duplikatów, ta selekcja przy dzisiejszym zakresie wychodzi identyczna liczebnie — wartość audytu jest tu **prewencyjna** (dokumentuje wykluczenia na wypadek poszerzenia korpusu ponad obecne 18) i procesowa (liver dostaje wreszcie tę samą rundę przeglądu duplikatów co kidney).

In [8]:
repaired_exclude_ids = sorted(existing_excluded_ids | new_findings)
print(f"exclude_dataset_ids: {len(existing_excluded_ids)} (existing) -> {len(repaired_exclude_ids)} (repaired)")

final_filters = {
    "organism": "Mouse",
    "organism_part": "Liver",
    "polarity": "Negative",
    "condition": "Wildtype",
    "mz_min": 200,
    "mz_max": 1400,
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
    "include_molecule_stats": True,
    "include_spatial_annotation_stats": False,  # see brain_dataset.ipynb section 6 for the cost rationale
    "exclude_dataset_ids": repaired_exclude_ids,
}
results_liver_repaired = explorer.filter(final_filters)
print(f"repaired liver shortlist: {len(results_liver_repaired)} datasets (previously {len(existing_selected_ids)})")
display(results_liver_repaired[["dataset_id", "name", "analyzer_type", "pixel_count", "molecule_count", "unique_molecule_count"]])

exclude_dataset_ids: 0 (existing) -> 5 (repaired)


METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

repaired liver shortlist: 18 datasets (previously 18)


,dataset_id,name,analyzer_type,pixel_count,molecule_count,unique_molecule_count
0,2025-08-19_19h29m48s,Lipids3_top_control_nodiverter,Orbitrap,511,207,5
1,2025-08-19_19h33m37s,Lipids4_top_Nh4F_diverter,Orbitrap,670,37,0
2,2025-08-19_19h33m18s,Lipids4_bottom_control_nodiverter,Orbitrap,681,109,6
3,2025-08-19_19h29m30s,Lipids3_bottom_Nh4F_diverter,Orbitrap,508,182,10
4,2025-08-19_19h28m58s,Lipids2_top_NH4F_diverter,Orbitrap,534,225,12
5,2025-08-19_19h23m41s,Lipids1_bottom_NH4F_diverter,Orbitrap,754,262,44
6,2025-08-19_19h27m06s,Lipids2_bottom_control_nodiverter,Orbitrap,532,120,1
7,2025-08-19_19h26m22s,Lipids1_top_control_nodiverter,Orbitrap,653,216,9
8,2025-06-30_15h03m07s,Tissue5_top_NH4F,Orbitrap,854,829,149
9,2025-06-30_15h02m48s,Tissue4_top_NH4F,Orbitrap,839,468,23


In [9]:
output_path = Path("data/liver_workspace/configs/datasets/liver_repaired")
exported = explorer.export_selection(output_path, sort_by="download_size_bytes", ascending=False)
exported

{'filters': PosixPath('data/liver_workspace/configs/datasets/liver_repaired/filter.json'),
 'selection': PosixPath('data/liver_workspace/configs/datasets/liver_repaired/selection.json')}

## Podsumowanie

- Szeroka pula (Wildtype, bez filtra m/z): 82 datasety.
- Obiektywne duplikaty techniczne: 3 klastry `high_confidence_duplicate` (8 rekordów → 3 zachowane, 5 wykluczonych) + 3 klastry `ambiguous_shared_template` (10 rekordów, **nie** wykluczone — w tym pouczający przypadek serii DKFZACLY, patrz sekcja 2).
- Wariant `null_mz_shift`: 0.
- Fragmenty mikroanatomiczne: 0 (heurystyka nie znalazła sygnału).
- Niska liczba pikseli (próg skalibrowany do liver, `<200`, nie `<500` z brain/kidney): 3, w większości pokrywające się z klastrem duplikatów z sekcji 2.
- Obecny, pobrany korpus 18 datasetów **nie zawiera** żadnego z tych problemów — audyt ma tu głównie wartość prewencyjną/procesową.
- Eksport do `data/liver_workspace/configs/datasets/liver_repaired/` — istniejący `liver/` nie został nadpisany.

Analiza wspólnego zakresu m/z między narządami — świadomie pominięta.